# Step 11 · Resolve Wikipedia references

Resolve Wikipedia reference links for the pathogen records selected in the saved V4 run. The association table is keyed by `pathogen_id`, preserving stable V4 taxonomy identity. Lookups checkpoint to CSV after each API batch and resume completed records on rerun. Only exact Wikipedia article resolution or exact GBIF species matches receive links.

Run `00_configure_run` and `03_acquire_pair_pubmed_records` first. The selected run is read from `data/v4/runs/<run-id>/resolved_config.json`; when the run selected pathogen groups, the full resolved pathogen selection is included. Set `FORCE_REFRESH = True` to repeat lookups for records in scope.

## Step 1 · Load the resolved run and taxonomy


In [1]:
import json, sys
from pathlib import Path
import pandas as pd
from IPython.display import display, Markdown

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'pathogen_atlas_v4').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'pathogen_atlas_v4').is_dir():
    raise RuntimeError('Open this notebook from the Pathogen Atlas V4 repository.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / 'notebooks'))
from _shared import configure_notebook, configured_run_id, staging_root_for_config
RUN_ID = configured_run_id(PROJECT_ROOT)
ROOT, LOG, LOG_PATH = configure_notebook('11_resolve_wikipedia_references', RUN_ID)
RUN_PATH = ROOT / 'data/v4/runs' / RUN_ID / 'resolved_config.json'
if not RUN_PATH.exists():
    raise FileNotFoundError(f'Run notebook 00 first; resolved selection is missing: {RUN_PATH}')
resolved_run = json.loads(RUN_PATH.read_text(encoding='utf-8'))
ACCEPTED_PATH = ROOT / 'data/v4/runs' / RUN_ID / 'release/pathogenic_interest_index.parquet'
if not ACCEPTED_PATH.exists():
    raise FileNotFoundError(f'Run notebook 05 first; accepted Explorer profiles are missing: {ACCEPTED_PATH}')
pathogen_ids = set(pd.read_parquet(ACCEPTED_PATH, columns=['pathogen_id']).pathogen_id.astype(str))
TAXONOMY_ROOT = staging_root_for_config(ROOT, resolved_run['source_config']) / 'taxonomy'
RUN_TAXONOMY_ROOT = ROOT / 'data/v4/runs' / RUN_ID / 'taxonomy'
PATHOGEN_TABLE = RUN_TAXONOMY_ROOT / 'pathogens.parquet'
if not PATHOGEN_TABLE.exists():
    raise FileNotFoundError('Run taxonomy was not resolved; run 00_configure_run.ipynb first.')
pathogens = pd.read_parquet(PATHOGEN_TABLE, columns=['pathogen_id','pathogen_group','accepted_name','rank'])
scope = pathogens.loc[pathogens.pathogen_id.astype(str).isin(pathogen_ids) & pathogens['rank'].astype(str).str.casefold().eq('species')]
pathogen_ids = set(scope.pathogen_id.astype(str))
PATHOGEN_GROUPS = sorted(scope.pathogen_group.astype(str).unique())
OUTPUT_PATH = ROOT / 'data/v4/references/release/11_resolve_wikipedia_references.csv'
FORCE_REFRESH = False
existing = pd.read_csv(OUTPUT_PATH, keep_default_na=False) if OUTPUT_PATH.exists() else pd.DataFrame()
display(Markdown(f"**Run:** `{RUN_ID}` · **Groups:** {', '.join(PATHOGEN_GROUPS)} · **Accepted species profiles:** {scope.pathogen_id.nunique():,} · **Existing rows:** {len(existing):,} · **Force refresh:** {FORCE_REFRESH}"))
LOG.info('Reference lookup prepared: run=%s groups=%s species=%d existing=%d', RUN_ID, PATHOGEN_GROUPS, scope.pathogen_id.nunique(), len(existing))


2026-11-03 16:45 INFO Notebook session started at 2026-11-03T16:45:59.705249+00:00
2026-11-03 16:45 INFO Project root: /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4
2026-11-03 16:45 INFO Run log: /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4/data/v4/logs/human-parasites-2026-11-03/11_resolve_wikipedia_references.log


**Run:** `human-parasites-2026-11-03` · **Groups:** parasites · **Accepted species profiles:** 348 · **Existing rows:** 3 · **Force refresh:** False

2026-11-03 16:45 INFO Reference lookup prepared: run=human-parasites-2026-11-03 groups=['parasites'] species=348 existing=3


## Step 2 · Resolve and save Wikipedia associations

The lookup writes one row per pathogen ID, saves progress after every request batch, and reuses completed rows on rerun.

In [ ]:
from pathogen_atlas_v4.references.links import resolve_wikipedia_links
service = 'Wikipedia'

links = resolve_wikipedia_links(PATHOGEN_TABLE, OUTPUT_PATH, PATHOGEN_GROUPS, force_refresh=FORCE_REFRESH, pathogen_ids=pathogen_ids)
selected_links = links.loc[links.pathogen_id.astype(str).isin(pathogen_ids)].copy()
found = selected_links.match_status.eq('found')
display(Markdown(f"**Accepted species for this run:** {len(pathogen_ids):,} · **{service} references found:** {int(found.sum()):,} · **No exact reference:** {int((~found).sum()):,} · **Cumulative catalog rows:** {len(links):,}"))
display(selected_links.head(12))
LOG.info('Reference lookup complete: service=Wikipedia selected_species=%d found_selected=%d cumulative_records=%d output=%s', len(pathogen_ids), int(found.sum()), len(links), OUTPUT_PATH)